<a href="https://colab.research.google.com/github/starsyntaxx/introduction-to-data-cleaning/blob/main/datasets/01_outliers/outliers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import files
uploaded = files.upload()

Saving listings.csv.gz to listings.csv.gz


In [4]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
df = pd.read_csv('listings.csv.gz')

In [11]:
df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2595,https://www.airbnb.com/rooms/2595,20260213082241,2026-02-14,city scrape,Skylit Studio Oasis | Midtown Manhattan Sanctuary,Prime Midtown | Spacious 500 Sq Ft | Pyramid S...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,2845,...,4.80,4.81,4.40,NaN,NaN,3,3,0,0,0.24
1,6848,https://www.airbnb.com/rooms/6848,20260213082241,2026-02-14,previous scrape,Only 2 stops to Manhattan studio,Comfortable studio apartment with super comfor...,NaN,https://a0.muscache.com/pictures/e4f031a7-f146...,15991,...,4.80,4.69,4.59,NaN,NaN,1,1,0,0,0.97
2,6872,https://www.airbnb.com/rooms/6872,20260213082241,2026-02-14,city scrape,Uptown Sanctuary w/ Private Bath (Month to Month),This charming distancing-friendly month-to-mon...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,16104,...,5.00,5.00,5.00,NaN,NaN,2,0,2,0,0.04
3,6990,https://www.airbnb.com/rooms/6990,20260213082241,2026-02-14,city scrape,UES Beautiful Blue Room,Beautiful peaceful healthy home,NaN,https://a0.muscache.com/pictures/45fb4ec7-6856...,16800,...,4.94,4.85,4.84,NaN,NaN,1,0,1,0,1.26
4,7097,https://www.airbnb.com/rooms/7097,20260213082241,2026-02-13,city scrape,"Perfect for Your Parents, With Garden & Patio",Parents/grandparents coming to town or are you...,NaN,https://a0.muscache.com/pictures/aaac19fc-4b4d...,17571,...,4.93,4.95,4.82,OSE-STRREG-0000008,NaN,2,0,2,0,2.16


In [7]:
df.isnull().sum()

,0
id,0
listing_url,0
scrape_id,0
last_scraped,0
source,0
...,...
calculated_host_listings_count,0
calculated_host_listings_count_entire_homes,0
calculated_host_listings_count_private_rooms,0
calculated_host_listings_count_shared_rooms,0


In [9]:
df['calculated_host_listings_count'].value_counts()

,count
calculated_host_listings_count,
1,17948
2,3976
3,1827
1210,1210
4,1128
...,...
53,53
25,50
44,44


In [24]:
data = np.array(df['calculated_host_listings_count'])

In [26]:
del len

In [27]:
N = len(data)

In [15]:
mean_y = np.mean(data)

print("Mean:", mean_y)

Mean: 69.9985457538757


In [16]:
std_y = np.std(data, ddof=1)

print("Standard Deviation:", std_y)

Standard Deviation: 228.94737442437525


In [17]:
deviations = np.abs(data - mean_y)

print(deviations)

[66.99854575 68.99854575 67.99854575 ... 68.99854575 68.99854575
 68.99854575]


In [18]:
max_deviations = np.max(deviations)

print("Maximum Deviation:", max_deviations)

Maximum Deviation: 1140.0014542461242


In [19]:
G_calculated = max_deviations / std_y

print("Grubbs Statistic:", G_calculated)

Grubbs Statistic: 4.979316566142512


In [32]:
outlier_index = np.argmax(deviations)

suspected_outlier = data[outlier_index]

print("Suspected Outlier:", suspected_outlier)

Suspected Outlier: 1210


In [20]:
alpha = 0.05

In [28]:
t_critical = stats.t.ppf(
    1 - alpha / (2 * N),
    N - 2
)

print("t Critical:", t_critical)

t Critical: 4.829845500278392


In [29]:
G_critical = (
    ((N - 1) / np.sqrt(N))
    * np.sqrt(
        t_critical**2 /
        (N - 2 + t_critical**2)
    )
)

print("Grubbs Critical Value:", G_critical)

Grubbs Critical Value: 4.828300435640068


In [34]:
if G_calculated > G_critical:
    print("Outlier Detected")
else:
    print("No Outlier Detected")

Outlier Detected


In [35]:
print("Suspected Outlier:", suspected_outlier)

Suspected Outlier: 1210


*Implementing Outlier detection by Parametric method: Assumes a Gaussian Bell Curve.*

In [36]:
#the mean defines the center of the curve, and the curve is mathematically smooth regardless of individual points
mu = np.mean(data)
#standard deviation of dataset from the mean
sigma = np.std(data, ddof=1)

print("Mean (mu):", mu)
print("Std (sigma):", sigma)

Mean (mu): 69.9985457538757
Std (sigma): 228.94737442437525


In [37]:
#How many standard deviations away is the values from the mean
z_scores = np.abs((data - mu) / sigma)

print(z_scores)

[0.29263732 0.30137295 0.29700513 ... 0.30137295 0.30137295 0.30137295]


In [38]:
threshold = 3

outliers = data[z_scores > threshold]

print("Outliers:", outliers)

Outliers: [1210 1210 1210 ... 1210 1210 1210]


*A Challenge of this method: masking: When large outliers exist and they skew the values of the mean, moving it away from the actal center. A remedy is Robust-Median Method.*

 - Smaller outliers might look normal

In [39]:
median = np.median(data)
print("Median:", median)

Median: 2.0


In [40]:
abs_deviation = np.abs(data - median)

In [41]:
mad = np.median(abs_deviation)

print("MAD:", mad)

MAD: 1.0


In [42]:
modified_z = 0.6745 * (data - median) / mad

print(modified_z)

[ 0.6745 -0.6745  0.     ... -0.6745 -0.6745 -0.6745]


In [43]:
threshold = 3.5

outliers = data[np.abs(modified_z) > threshold]

print("Outliers:", outliers)

Outliers: [ 8 29 29 ... 27 27 17]


In [ ]:
# Using IQR fofr outlier detection

In [44]:
Q1 = np.percentile(data, 25)
Q3 = np.percentile(data, 75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = data[(data < lower_bound) | (data > upper_bound)]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Outliers:", outliers)

Q1: 1.0
Q3: 9.0
IQR: 8.0
Lower bound: -11.0
Upper bound: 21.0
Outliers: [29 29 29 ... 27 27 27]


In [60]:
from sklearn.ensemble import IsolationForest

X = df.values  # your dataset

In [61]:
model = IsolationForest(
    n_estimators=100,
    contamination=0.05,  # expected % of outliers
    random_state=42
)

In [62]:
model.fit(X)

ValueError: could not convert string to float: 'https://www.airbnb.com/rooms/2595'